In [ ]:
"""
Refined: ResNet-based Semantic Segmentation on Pascal VOC (clean, modular, metrics, amp, checkpointing)
- Handles masks properly (no weird *255 hacks).
- Uses safe nearest interpolation for masks.
- Computes mIoU + pixel accuracy.
- Mixed precision training (AMP) and checkpointing of best mIoU.
- Simple decoder using bilinear upsample + convs (stable).
"""

import os
import random
from pathlib import Path
from typing import Tuple, Dict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
import matplotlib.pyplot as plt

# ----------------------------
# Config / hyperparameters
# ----------------------------
SEED = 42
DATA_ROOT = "./data"
BATCH_SIZE = 8
NUM_WORKERS = min(8, os.cpu_count() or 1)
NUM_CLASSES = 21
IMAGE_SIZE = (224, 224)  # (H, W)
NUM_EPOCHS = 50
LR = 1e-3
WEIGHT_DECAY = 1e-4
CHECKPOINT_PATH = "resnet_seg_best.pth"
PRINT_FREQ_BATCH = 200  # batches
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = True if DEVICE.type == "cuda" else False

# ----------------------------
# Determinism
# ----------------------------
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

# ----------------------------
# Transforms (images vs masks)
# ----------------------------
# Image augmentations/processing
train_img_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_img_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Mask transform: convert PIL mask -> LongTensor (H, W) with original integer labels (0..20, 255)
def mask_to_tensor(pic: Image.Image) -> torch.LongTensor:
    # Keeps the 255 ignore label as-is
    arr = np.array(pic, dtype=np.uint8)  # shape (H,W)
    return torch.from_numpy(arr.astype(np.int64))  # LongTensor

train_mask_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE, interpolation=Image.NEAREST),
    transforms.Lambda(mask_to_tensor)
])

val_mask_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE, interpolation=Image.NEAREST),
    transforms.Lambda(mask_to_tensor)
])

# ----------------------------
# Datasets & loaders
# ----------------------------
print("Loading Pascal VOC 2012 (this may download if not present)...")
train_dataset = torchvision.datasets.VOCSegmentation(
    root=DATA_ROOT,
    year="2012",
    image_set="train",
    download=True,
    transform=train_img_transform,
    target_transform=train_mask_transform
)

val_dataset = torchvision.datasets.VOCSegmentation(
    root=DATA_ROOT,
    year="2012",
    image_set="val",
    download=True,
    transform=val_img_transform,
    target_transform=val_mask_transform
)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

# ----------------------------
# Simple ResNet backbone + decoder
# ----------------------------
class ResNetSeg(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES, pretrained_backbone: bool = True):
        super().__init__()
        # load backbone (ResNet18) and keep all conv layers (up to layer4)
        try:
            # modern torchvision API (weights)
            resnet = torchvision.models.resnet18(weights=(torchvision.models.ResNet18_Weights.DEFAULT if pretrained_backbone else None))
        except Exception:
            # fallback older API
            resnet = torchvision.models.resnet18(pretrained=pretrained_backbone)

        self.backbone = nn.Sequential(*list(resnet.children())[:-2])  # outputs (N,512,H/32,W/32)

        # a light decoder: repeated bilinear upsample + conv
        # decoder channels progressively reduce -> num_classes
        self.decoder = nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            # upsample x2
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            # upsample x2
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(128, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            # upsample x2
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(64, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            # upsample x2 -> back to IMAGE_SIZE (224)
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, num_classes, kernel_size=1)
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.decoder.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1.)
                nn.init.constant_(m.bias, 0.)

    def forward(self, x):
        feat = self.backbone(x)  # (N,512,H/32,W/32)
        out = self.decoder(feat)  # (N,num_classes,H,W)
        # ensure out spatial size matches input (safety)
        if out.shape[-2:] != x.shape[-2:]:
            out = F.interpolate(out, size=x.shape[-2:], mode='bilinear', align_corners=False)
        return out


# ----------------------------
# Metrics: confusion matrix and IoU
# ----------------------------
def fast_confusion_matrix(preds: torch.Tensor, labels: torch.Tensor, num_classes: int, ignore_index: int = 255):
    """
    preds: (N,H,W) predicted class indices (int)
    labels: (N,H,W) ground truth (int) with possible ignore_index
    returns: (num_classes, num_classes) confusion matrix where row = gt, col = pred
    """
    mask = (labels != ignore_index)
    preds = preds[mask]
    labels = labels[mask]
    if preds.numel() == 0:
        return torch.zeros((num_classes, num_classes), dtype=torch.long)

    k = (labels * num_classes + preds).long()
    bincount = torch.bincount(k, minlength=num_classes * num_classes)
    conf = bincount.reshape(num_classes, num_classes)
    return conf

def compute_miou(conf_matrix: torch.Tensor):
    # conf_matrix: (C,C): rows gt, cols pred
    true_positive = conf_matrix.diag().float()
    false_positive = conf_matrix.sum(dim=0).float() - true_positive
    false_negative = conf_matrix.sum(dim=1).float() - true_positive
    denom = true_positive + false_positive + false_negative
    iou = true_positive / (denom + 1e-9)
    miou = torch.mean(iou[~torch.isnan(iou)])  # ignore NaNs when a class never appears
    return miou.item(), iou.cpu().numpy()

# ----------------------------
# Training & validation loops (with AMP)
# ----------------------------
def train_one_epoch(model, dataloader, optimizer, criterion, device, scaler):
    model.train()
    running_loss = 0.0
    pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc="Train", leave=False)
    for batch_idx, (images, masks) in pbar:
        images = images.to(device, non_blocking=PIN_MEMORY)
        # masks: (N,H,W) LongTensor already
        masks = masks.to(device, non_blocking=PIN_MEMORY)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=scaler is not None):
            outputs = model(images)  # (N,C,H,W)
            # outputs and masks must match spatial size; we've ensured that in forward
            loss = criterion(outputs, masks)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)  # optional
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            optimizer.step()

        running_loss += loss.item()
        if (batch_idx + 1) % PRINT_FREQ_BATCH == 0:
            pbar.set_postfix(loss=running_loss / (batch_idx + 1))
    return running_loss / len(dataloader)


def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    conf_matrix = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long, device=device)

    with torch.no_grad():
        pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc="Val", leave=False)
        for batch_idx, (images, masks) in pbar:
            images = images.to(device, non_blocking=PIN_MEMORY)
            masks = masks.to(device, non_blocking=PIN_MEMORY)

            outputs = model(images)  # (N,C,H,W)
            loss = criterion(outputs, masks)
            running_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)  # (N,H,W)
            conf = fast_confusion_matrix(preds.cpu(), masks.cpu(), NUM_CLASSES, ignore_index=255)
            conf_matrix += conf.to(device)

    miou, per_class_iou = compute_miou(conf_matrix.cpu())
    pixel_acc = conf_matrix.diag().sum().float() / conf_matrix.sum().float()
    return running_loss / len(dataloader), miou, pixel_acc.item(), per_class_iou


# ----------------------------
# Visual helpers
# ----------------------------
VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus',
    'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike',
    'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

def create_colormap():
    colors = [
        [0, 0, 0], [128, 0, 0], [0, 128, 0], [128, 128, 0], [0, 0, 128],
        [128, 0, 128], [0, 128, 128], [128, 128, 128], [64, 0, 0], [192, 0, 0],
        [64, 128, 0], [192, 128, 0], [64, 0, 128], [192, 0, 128], [64, 128, 128],
        [192, 128, 128], [0, 64, 0], [128, 64, 0], [0, 192, 0], [128, 192, 0],
        [0, 64, 128]
    ]
    return np.array(colors, dtype=np.uint8)

COLORMAP = create_colormap()

def visualize_sample_from_dataset(dataset, idx=0):
    """Show image, ground truth mask (colored)"""
    img, mask = dataset[idx]  # img: tensor normalized, mask: LongTensor (H,W)
    # Denormalize image
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    if isinstance(img, torch.Tensor):
        img_denorm = img * std + mean
        img_np = img_denorm.permute(1,2,0).numpy().clip(0,1)
    else:
        img_np = np.array(img)/255.0

    mask_np = mask.numpy().astype(np.int32)
    mask_vis = np.where(mask_np == 255, 0, mask_np)  # show ignore as background
    colored = COLORMAP[mask_vis]

    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1); plt.imshow(img_np); plt.title("Image"); plt.axis('off')
    plt.subplot(1,2,2); plt.imshow(colored); plt.title("Mask"); plt.axis('off')
    plt.show()


def visualize_predictions(model, dataset, device, num_samples=3):
    model.eval()
    with torch.no_grad():
        for i in range(num_samples):
            img, mask = dataset[i]
            inp = img.unsqueeze(0).to(device)
            out = model(inp)
            pred = torch.argmax(out, dim=1).squeeze(0).cpu().numpy()
            mask_np = mask.numpy()
            img_denorm = (img * torch.tensor([0.229,0.224,0.225]).view(3,1,1) + torch.tensor([0.485,0.456,0.406]).view(3,1,1)).permute(1,2,0).numpy().clip(0,1)
            gt_vis = np.where(mask_np==255, 0, mask_np)
            plt.figure(figsize=(12,4))
            plt.subplot(1,3,1); plt.imshow(img_denorm); plt.title("Image"); plt.axis('off')
            plt.subplot(1,3,2); plt.imshow(COLORMAP[gt_vis]); plt.title("GT"); plt.axis('off')
            plt.subplot(1,3,3); plt.imshow(COLORMAP[pred]); plt.title("Pred"); plt.axis('off')
            plt.show()


# ----------------------------
# Main training orchestration
# ----------------------------
def main():
    model = ResNetSeg(num_classes=NUM_CLASSES, pretrained_backbone=True).to(DEVICE)
    print(f"Model on device: {DEVICE}, params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    criterion = nn.CrossEntropyLoss(ignore_index=255)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.1)

    scaler = torch.cuda.amp.GradScaler() if DEVICE.type == "cuda" else None

    best_miou = 0.0
    train_losses, val_losses, val_mious = [], [], []

    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, scaler)
        val_loss, val_miou_epoch, val_pixacc, per_class_iou = validate(model, val_loader, criterion, DEVICE)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_mious.append(val_miou_epoch)

        scheduler.step()

        print(f"Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | Val mIoU: {val_miou_epoch:.4f} | PixelAcc: {val_pixacc:.4f}")

        # checkpoint
        if val_miou_epoch > best_miou:
            best_miou = val_miou_epoch
            torch.save({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "miou": best_miou,
            }, CHECKPOINT_PATH)
            print(f"Saved best model (mIoU={best_miou:.4f}) -> {CHECKPOINT_PATH}")

    print("\nTraining finished.")
    print(f"Best Val mIoU: {best_miou:.4f}")

    # visualize a few
    visualize_predictions(model, val_dataset, DEVICE, num_samples=3)


if __name__ == "__main__":
    main()
